In [1]:
import os

train_dir = './coco/images/train/'
seg_train = './coco/segmentations/seg_train/'
val_dir = './coco/images/val/'
seg_val = './coco/segmentations/seg_val/'

print('Train contains:', len(os.listdir(train_dir)), 'images')
print('Seg_train contains:',len(os.listdir(seg_train)), 'images')
print('Validation contains:',len(os.listdir(val_dir)), 'images')
print('Seg_val contains:',len(os.listdir(seg_val)), 'images')
#print('Test contains:',len(os.listdir('./coco//content/test/')), 'images')

Train contains: 45623 images
Seg_train contains: 45623 images
Validation contains: 3200 images
Seg_val contains: 1158 images


In [4]:
## Intent seguint el tutorial

In [8]:
import matplotlib.pyplot as plt
import cv2
import os
import torch
from PIL import Image
import numpy as np
from torch.utils.data import Dataset
from pycocotools.coco import COCO

if torch.cuda.is_available():
    device=torch.device("cuda")
else:
    device=torch.device("cpu")

device=torch.device("cpu") #temporal
print(device)
"""
Output must be--> device(type='cuda')
"""

cpu


"\nOutput must be--> device(type='cuda')\n"

In [10]:
from torchvision.transforms import ToTensor
from torch.utils.data import DataLoader

In [11]:
class FashionDataset(Dataset):
    def __init__(self, img_dir, mask_dir, annotation_file, target_height=768, target_width=512, transform=None, originals=False):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.coco = COCO(annotation_file)
        self.transform = transform
        self.target_height = target_height
        self.target_width = target_width
        mask_basenames = [basename.split("_")[0] for basename in os.listdir(mask_dir)]
        basenames = [base.split(".")[0] for base in os.listdir(img_dir) if base.split(".")[0] in mask_basenames]
        name_img_files = [basename +'.jpg' for basename in basenames]
        mask_img_files = [basename +'_seg.png' for basename in basenames]
        self.img_files = sorted(name_img_files)
        self.mask_files = sorted(mask_img_files)
        self.originals = originals

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_files[idx])
        mask_path = os.path.join(self.mask_dir, self.mask_files[idx])
        image_orig = Image.open(img_path).convert('RGB')
        mask_orig = Image.open(mask_path)

        w, h = image_orig.size
        scale = min(self.target_width/w, self.target_height/h) #keep aspect ratio
        new_w = int(w*scale)
        new_h = int(h*scale)

        #resize keeping aspect ratio
        image =image_orig.resize((new_w, new_h), Image.BILINEAR)
        mask =mask_orig.resize((new_w, new_h), Image.NEAREST)
        #create empty images
        padded_image =Image.new("RGB", (self.target_width, self.target_height), (0, 0, 0))
        padded_mask =Image.new("L", (self.target_width, self.target_height), 0)
        #center position
        x_offset =(self.target_width - new_w)//2
        y_offset =(self.target_height - new_h)//2
        padded_image.paste(image,(x_offset, y_offset))
        padded_mask.paste(mask,(x_offset, y_offset))
        
        mask = np.array(padded_mask)
        if mask.ndim == 3:
            mask = mask[:,:,0]
        if self.transform:
            image = self.transform(padded_image)
        else:
            image = ToTensor()(padded_image)
        mask = torch.as_tensor(mask, dtype=torch.long)
        if self.originals:
            return image_orig, mask_orig, image, mask
        return image, mask

 
# Transform PIL image --> PyTorch tensor
def get_transform():
    return ToTensor()
 
# Load training dataset
train_dataset = FashionDataset(
    img_dir=train_dir,
    mask_dir=seg_train,
    annotation_file = "./coco/annotations/instances_attributes_train2020.json"
    #transform=get_transform()  # define this if needed
)
 
# Load validation dataset
valid_dataset = FashionDataset(
    img_dir=val_dir,
    mask_dir=seg_val,
    annotation_file = "./coco/annotations/instances_attributes_val2020.json"
    #transforms=get_transform()
)
 
# Load dataset with DataLoaders, DataLoader is used to load data in batches.
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
val_loader = DataLoader(valid_dataset, batch_size=1, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))

loading annotations into memory...
Done (t=4.96s)
creating index...
index created!
loading annotations into memory...
Done (t=0.14s)
creating index...
index created!


In [12]:
import torchvision
from torchvision.models.detection import MaskRCNN
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
 
# Load a pre-trained Mask R-CNN model with a ResNet-50 FPN backbone
model = torchvision.models.detection.maskrcnn_resnet50_fpn(pretrained=True)
 
# Determine the number of classes (including background) from training dataset
# COCO format includes category IDs, and we add +1 for background
num_classes = len(train_dataset.coco.getCatIds()) + 1
 
# Replace the existing box predictor with a new one for our number of classes
# in_features_box: number of input features to the classification layer
in_features_box = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features_box, num_classes)
 
# Replace the existing mask predictor with a new one for our number of classes
# in_features_mask: number of input channels to the first convolutional layer 
in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
hidden_layer = 256
model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, hidden_layer, num_classes)
 
# Move the model to the specified device (e.g., GPU) for training or inference
model.to(device)

/mnt/a27b1cbf-298a-4896-8528-84006d36fd9e/Nero/Documents/Acadèmic/UPC - MAMME/OR/P2/OR_task2/venv/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/mnt/a27b1cbf-298a-4896-8528-84006d36fd9e/Nero/Documents/Acadèmic/UPC - MAMME/OR/P2/OR_task2/venv/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MaskRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


MaskRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=0.0)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=0.0)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=0.0)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=0.0)
          (relu): ReLU(in

In [13]:
# Get parameters that require gradients (the model's trainable parameters)
params = [p for p in model.parameters() if p.requires_grad]
 
# Define the optimizer SGD(Stochastic Gradient Descent) 
optimizer = torch.optim.SGD(params, lr=0.005,
                            momentum=0.9, weight_decay=0.0005)

In [27]:
import utils
#SOBREESCRIC "train_one_epoch" per poder anar canviant coses sense haver de reiniciar el kernel cada vegada
def train_one_epoch(model, optimizer, data_loader, device, epoch, print_freq, scaler=None):
    model.train()
    metric_logger = utils.MetricLogger(delimiter="  ")
    metric_logger.add_meter("lr", utils.SmoothedValue(window_size=1, fmt="{value:.6f}"))
    header = f"Epoch: [{epoch}]"

    lr_scheduler = None
    if epoch == 0:
        warmup_factor = 1.0 / 1000
        warmup_iters = min(1000, len(data_loader) - 1)

        lr_scheduler = torch.optim.lr_scheduler.LinearLR(
            optimizer, start_factor=warmup_factor, total_iters=warmup_iters
        )

    for images, targets in metric_logger.log_every(data_loader, print_freq, header):
        images = list(image.to(device) for image in images)
        #peta aquí, no està pillant les targets bé. Potser el isinstance??
        targets = [{k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in t.items()} for t in targets]
        with torch.cuda.amp.autocast(enabled=scaler is not None):
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())

        # reduce losses over all GPUs for logging purposes
        loss_dict_reduced = utils.reduce_dict(loss_dict)
        losses_reduced = sum(loss for loss in loss_dict_reduced.values())

        loss_value = losses_reduced.item()

        if not math.isfinite(loss_value):
            print(f"Loss is {loss_value}, stopping training")
            print(loss_dict_reduced)
            sys.exit(1)

        optimizer.zero_grad()
        if scaler is not None:
            scaler.scale(losses).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            losses.backward()
            optimizer.step()

        if lr_scheduler is not None:
            lr_scheduler.step()

        metric_logger.update(loss=losses_reduced, **loss_dict_reduced)
        metric_logger.update(lr=optimizer.param_groups[0]["lr"])

    return metric_logger

In [28]:
#from engine import train_one_epoch, evaluate
from engine import evaluate

# Number of epochs for training
num_epochs = 3
 
# Loop through each epoch
for epoch in range(num_epochs):
    print(f"\nEpoch {epoch + 1}/{num_epochs}")
 
    # Train the model for one epoch, printing status every 25 iterations
    train_one_epoch(model, optimizer, train_loader, device, epoch, print_freq=25)  # Using train_loader for training
 
    # Evaluate the model on the validation dataset
    evaluate(model, val_loader, device=device)  # Using val_loader for evaluation
 
    # Optionally, save the model checkpoint after each epoch
    torch.save(model.state_dict(), f"model_epoch_{epoch + 1}.pth")


Epoch 1/3
1.99896240234375


AttributeError: 'Tensor' object has no attribute 'items'

In [ ]:
## PROVANT MAIN DEL GH AMB QUALSEVOL MODEL

In [3]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import OneCycleLR
from dataset import FashionDataset
import config
from torch.utils.data import random_split
import torchvision.transforms as T
from Segformer import segformer_mit_b3
from deeplabv3plus import deeplabv3plus
from utils import (get_dataloaders, train_validate_model, evaluate_model, meanIoU, visualize_predictions)
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
import matplotlib.pyplot as plt
import random
import time

FULL_CLASSES = ('shirt, blouse','top, t-shirt, sweatshirt','sweater','cardigan','jacket','vest','pants','shorts','skirt','coat','dress','jumpsuit','cape','glasses','hat','headband, head covering, hair accessory','tie','glove','watch','belt','leg warmer','tights, stockings','sock','shoe','bag, wallet','scarf','umbrella','hood','collar','lapel','epaulette','sleeve','pocket','neckline','buckle','zipper','applique','bead','bow','flower','fringe','ribbon','rivet','ruffle','sequin','tassel')
MAIN_CLASSES = FULL_CLASSES[:27]
NUM_CLASSES = len(MAIN_CLASSES) + 1 #+1 for background

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

def build_transforms():
    return T.Compose([T.ToTensor(),T.Normalize(mean=[0.485, 0.456, 0.406],std=[0.229, 0.224, 0.225])])

def build_datasets(target_height, target_width, transform):
    train_set_full = FashionDataset(config.TRAIN_IMG,config.TRAIN_MASK,target_height,target_width,transform)
    test_set = FashionDataset(config.TEST_IMG,config.TEST_MASK,target_height,target_width,transform)

    val_size = len(test_set)
    if val_size > len(train_set_full):
        raise ValueError(f"Validation size ({val_size}) is larger than train set ({len(train_set_full)}).")

    train_size = len(train_set_full) - val_size
    generator = torch.Generator().manual_seed(42)
    train_set, val_set = random_split(train_set_full,[train_size, val_size],generator=generator)
    return train_set, val_set, test_set

def build_model(model_name: str, num_classes: int, device: torch.device, pretrained: bool = True):
    if model_name == "deeplabv3+":
        model = deeplabv3plus(num_classes)
    elif model_name == "segformer":
        model = segformer_mit_b3(in_channels=3, num_classes=num_classes)
        if pretrained:
            weights_path = "segformers/segformer_mit_b3_imagenet_weights.pt"
            if os.path.exists(weights_path):
                state_dict = torch.load(weights_path, map_location=device)
                model.backbone.load_state_dict(state_dict)
            else:
                print(f"Warning: pretrained weights not found at {weights_path}")
    elif model_name == "maskRCNN":
        raise NotImplementedError("maskRCNN branch is not implemented yet.")
    else:
        raise ValueError(f"Unknown model_name: {model_name}")
    return model.to(device)

def get_id_to_color():
    return np.array([
        [0,0,0],
        [255,0,0],
        [0,255,0],
        [0,0,255],
        [255,255,0],
        [255,0,255],
        [0,255,255],
        [128,0,0],
        [0,128,0],
        [0,0,128],
        [128,128,0],
        [128,0,128],
        [0,128,128],
        [192,64,0],
        [64,192,0],
        [0,64,192],
        [192,0,64],
        [64,0,192],
        [0,192,64],
        [255,128,0],
        [255,0,128],
        [128,255,0],
        [0,255,128],
        [128,0,255],
        [0,128,255],
        [255,128,128],
        [128,255,128],
        [128,128,255]
    ], dtype=np.uint8)

def main():
    set_seed(42)

    target_width = 192
    target_height = 192
    n_epochs = 20
    base_lr = 5e-05
    batch_size = 5
    pretrained = True
    data_augmentation = False
    model_name = "deeplabv3+"
    model_file = f"{model_name}_{target_height}_{target_width}_{base_lr}_{batch_size}"
    suffixes = []
    if pretrained:
        suffixes.append("pretrained")
    if data_augmentation:
        suffixes.append("data_aug")
    if suffixes:
        model_file += "_" + "_".join(suffixes)

    device = get_device()
    device=torch.device("cpu") #override!! no em pilla la GPU
    print(f"Using device: {device}")

    transform = build_transforms()
    train_set, val_set, test_set = build_datasets(target_height,target_width,transform)
    train_loader, val_loader, test_loader = get_dataloaders(train_set, val_set, test_set, batch_size=batch_size)
    model = build_model(model_name, NUM_CLASSES, device, pretrained=pretrained)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=base_lr, weight_decay=1e-4)
    #scheduler = OneCycleLR(optimizer, max_lr= max_lr, epochs = n_epochs,steps_per_epoch = len(train_loader), pct_start=0.3, div_factor=10, anneal_strategy='cos')
    warmup_epochs = 2

    warmup_scheduler = LinearLR(
        optimizer,
        start_factor=0.1,
        total_iters=warmup_epochs * len(train_loader)
    )

    cosine_scheduler = CosineAnnealingLR(
        optimizer,
        T_max=(n_epochs - warmup_epochs) * len(train_loader),
        eta_min=1e-6
    )

    scheduler = SequentialLR(
        optimizer,
        schedulers=[warmup_scheduler, cosine_scheduler],
        milestones=[warmup_epochs * len(train_loader)]
    )
    results = {}
    start_train_val = time.time()
    _ = train_validate_model(model, n_epochs, model_name, criterion, optimizer, 
                         device, train_loader, val_loader,NUM_CLASSES, lr_scheduler = scheduler, output_path = config.MODELS, model_name_save=model_file)
    results['train_time'] = time.time() - start_train_val
    model_path_best_loss = os.path.join(config.MODELS, f"{model_file}_best_loss.pt")
    if not os.path.exists(model_path_best_loss):
        raise FileNotFoundError(f"Checkpoint not found for best loss: {model_path_best_loss}")
    model.load_state_dict(torch.load(model_path_best_loss, map_location=device))
    test_metrics = evaluate_model(model, test_loader, criterion, NUM_CLASSES, device)
    print(f"\nModel has {test_metrics['mDice']} mean Dice in test set")
    result_best_loss = {
            "mIoU": float(test_metrics["mIoU"]),
            "mDice": float(test_metrics["mDice"]),
            "mDice_no_bg": float(test_metrics["mDice_no_bg"]),
            "accuracy": float(test_metrics["accuracy"]),
            "mean_acc": float(test_metrics["mean_acc"]),
            "mean_acc_no_bg": float(test_metrics["mean_acc_no_bg"]),
            "dice_per_class": test_metrics["dice_per_class"].tolist(),
            "accuracy_per_class": test_metrics["accuracy_per_class"].tolist(),
            "iou_per_class": test_metrics["iou_per_class"].tolist(),
        }
    results['best loss model'] = result_best_loss


    model_path_mdice = os.path.join(config.MODELS, f"{model_file}_best_mDice.pt")
    if not os.path.exists(model_path_mdice):
        raise FileNotFoundError(f"Checkpoint not found for mDice: {model_path_mdice}")
    model.load_state_dict(torch.load(model_path_mdice, map_location=device))
    test_metrics = evaluate_model(model, test_loader, criterion, NUM_CLASSES, device)
    print(f"\nModel has {test_metrics['mDice']} best mean Dice in test set")
    result_mdice = {
            "mIoU": float(test_metrics["mIoU"]),
            "mDice": float(test_metrics["mDice"]),
            "mDice_no_bg": float(test_metrics["mDice_no_bg"]),
            "accuracy": float(test_metrics["accuracy"]),
            "mean_acc": float(test_metrics["mean_acc"]),
            "mean_acc_no_bg": float(test_metrics["mean_acc_no_bg"]),
            "dice_per_class": test_metrics["dice_per_class"].tolist(),
            "accuracy_per_class": test_metrics["accuracy_per_class"].tolist(),
            "iou_per_class": test_metrics["iou_per_class"].tolist(),
        }
    results['mDice model'] = result_mdice
    metrics_json_path = os.path.join(config.RESULTS, f"{model_file}_test_metrics.json")
    with open(metrics_json_path, "w") as f:
        json.dump(results, f, indent=4)


    #id_to_color = np.array([[i,i,i] for i in range(28)], dtype=np.uint8)
    id_to_color = get_id_to_color()
    num_test_samples = 10
    _, axes = plt.subplots(num_test_samples, 3, figsize=(3*6, num_test_samples * 4))
    visualize_predictions(model, test_set, axes, device, numTestSamples=num_test_samples, 
                        id_to_color = id_to_color, model_name_save = model_file)

if __name__ == '__main__':
    main()

Using device: cpu
Starting 1 epoch ...


  0%|                                                  | 0/8893 [00:01<?, ?it/s]


IndexError: Target 28 is out of bounds.